# Week 12 — Capstone — a privacy-era measurement stack

**Goal.** Wire eleven weeks into one system and tell the story end to end.

**Deliverable.** A polished repo, a README that reads like an engineering post, and the blog post itself.

**Rough shape of the week.** No new reading. 8h assembling and writing.

---
### Ground rules (they apply every week)

1. **Beat a dumb baseline or it didn't happen.** Logistic regression or the global mean.
   Log the baseline in the same table as the fancy model.
2. **Split by time, never at random.** `split.time_split` — and call
   `split.check_no_leakage` so the assertion, not your memory, enforces it.
3. **Log every run** with `registry.log_result(...)`, including the ones that lost.
   The losing runs are what make the write-up honest.
4. **Write the finding down** in this week's `README.md` while it is fresh.

### Reading

PDFs are in `papers/` next to this notebook — see `papers/README.md`.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from adslab import data, metrics, plots, split, registry, encoders, calibration

plots.use_style()
pd.set_option("display.width", 140, "display.max_columns", 60)
print("harness ready")

## The system

```
      impressions (simulated traffic, privacy-induced label loss)
                        |
        [W5/W10]  consent + signal loss applied
                        |
        [W4]  delayed-feedback-corrected CVR model
                        |
        [W3]  calibration layer
                        |
        [W6]  attribution -> value per impression
                        |
        [W8]  auction + bid shading
                        |
        [W9]  pacing + target-CPA controller
                        |
                  spend, conversions, realised CPA
```

Each stage exists in a previous week. This week is integration and narrative, not new
modelling. Resist the urge to improve a component — the value is in the seams.

In [ ]:
import sys; sys.path.insert(0, "..")
from adslab import data, metrics, plots, split, registry
plots.use_style()

print(registry.to_markdown())   # everything you have built, in one table

## 1. The pipeline object

Wrap each stage behind one interface so the whole thing can be run with components
switched on and off. The ablation is the experiment: run the stack with each correction
disabled and measure what the business metric does.

| configuration | realised CPA vs target | conversions measured vs true |
|---|---|---|
| everything off (naive) | | |
| + calibration | | |
| + delayed-feedback correction | | |
| + conversion modeling | | |
| full stack | | |

That table is the capstone's headline result.

In [ ]:
class MeasurementStack:
    def __init__(self, cvr_model, calibrator=None, delay_model=None,
                 attribution=None, shader=None, pacer=None):
        # TODO: each stage optional so you can ablate
        raise NotImplementedError

    def run_day(self, traffic):
        raise NotImplementedError

## 2. Simulated traffic with everything wrong at once

Real conditions, all at the same time: delayed conversions, 30% consent loss,
non-stationary traffic, and a competitive auction. Individually each was survivable.
The question is whether the corrections compose or interfere — and interference is a
finding, not a bug in your write-up.

In [ ]:
# TODO

## 3. The ablation

Run every configuration, fill in the table, plot the cumulative CPA error over a
simulated month per configuration.

In [ ]:
# fig, ax = plt.subplots()
# ... one line per configuration
# print(plots.save(fig, 12, "ablation_cpa_error"))

## 4. Write the post

Structure that works:

1. **The problem, in money.** Privacy loss makes conversions invisible; invisible
   conversions make bids wrong; wrong bids waste budget. One paragraph, no jargon.
2. **What I built and on what data.** Emphasise it is all public data — that is what
   makes it checkable, and checkable is the whole point.
3. **Four findings with plots.** Pick the four best from twelve weeks. Suggested:
   the calibration-breaks-under-label-loss plot (W3), the delayed-feedback bias by
   elapsed time (W4), the recovery-vs-consent-loss curve (W5), and the
   overspend-vs-delay curve (W9).
4. **What I got wrong.** The negative results — where the sequence model did not help,
   where correction stopped working. This section is what separates it from content
   marketing, and it is the section a good interviewer will want to talk about.
5. **What I would build next.**

Then the repo README: the same story, shorter, with the reproduction instructions that
actually work on a clean clone. Test that claim by following them yourself.

In [ ]:
# print(registry.to_markdown())  # the full twelve-week table for the post

---
## Log the results

Every model you tried, including the baseline and including the failures. `notes` is the
one sentence you would say out loud about the run — future-you assembles the write-up
from these, so write it now while you still remember why the run mattered.

In [ ]:
# registry.log_result(
#     week=12,
#     model="lightgbm_hashed_2^18",
#     metrics=metrics.evaluate(y_test, p_test),
#     dataset="attribution",
#     params=dict(n_bits=18, num_leaves=63, lr=0.05),
#     notes="beats LR by 0.011 AUC; most of the gain is from cat3 x cat7 interactions",
# )

print(registry.to_markdown(week=12))

---
## Write it up

Open `README.md` in this folder and fill in the three sections. Keep it to a page.

- **What I built** — one paragraph, no code.
- **What the numbers say** — paste the table above; say which comparison is the honest one.
- **What surprised me** — the part worth reading. If nothing surprised you, you probably
  did not stress the model hard enough.

Then commit:

```bash
git add week12_* results/
git commit -m "week 12: <the finding, not the task>"
```